# Module 5 — Skills + SODA Quality Checks

In M4 you built `CurveValidator` with single-record validation.
In M5 you add data quality checks — generated with a reusable Copilot skill.

> **⏱ 30 minutes**

> **Prerequisite**: `curve_pipeline/validator.py` must be implemented and green from M4.


---
## Three levels of reusable context

You have now used three different ways to give Copilot persistent context:

| Where it lives | When it applies | Good for |
|---|---|---|
| `.github/copilot-instructions.md` | Every prompt automatically | Project-wide rules, domain vocabulary |
| `.github/prompts/<name>.prompt.md` | When you invoke `/name` | Project-specific repeatable tasks |
| `.github/skills/<name>.md` | When you invoke `/name` | Project-**agnostic** tasks — reusable across repos |

The difference between a prompt and a skill: a prompt knows your column names, your file paths, your business rules — it is project-specific by design. A skill reads the schema from whatever file you point it at and generates something that fits. You could copy it to any pipeline repo and `/add-soda-checks` would work there too.

---
## What is SODA?

[SODA](https://docs.soda.io/) (Soda Data Quality) is a data quality framework: you declare checks in a YAML file and run them against a dataset. Checks can assert things like "this column has no nulls", "this column only contains these values", "row count is above a threshold".

In this pipeline, SODA checks are the **last line of defence before a curve is published** — they catch structural problems (null CET timestamps, unknown types, non-numeric quotes) that the regression runner wouldn't catch because they are about data shape, not CET conversion correctness.

> **Discussion**: The regression runner checks that `cetStarttime` has the right *value*. SODA checks that it is *present*. Would a SODA `missing_count = 0` check on `cetStarttime` have caught the copy-through bug?

---
## Step 1 — Generate SODA checks with `/add-soda-checks`

This is a **skill** stored in `.github/skills/add-soda-checks.md`. It is project-agnostic — it reads your schema from whatever loader file you point it at.

**Use this skill in Copilot Chat:**



In [ ]:
'''
/add-soda-checks
'''


Point it at `curve_pipeline/curve_loader.py` for the schema. Accept the generated `soda_checks/curve_quality.yml`.

Make sure it includes at minimum:
- `missing_count` = 0 for `cetStartDate` and `cetStarttime`
- `valid_values` for `type`: (`CALCULATED`, `INPUT`)
- `valid_values` for `order`: (`A`, `B`)
- `quote` is numeric

**Note**: Do NOT add a `min > 0` check on `quote` — negative prices are valid in power markets.

---
## Step 2 — Implement `run_soda_checks`

Now wire the SODA checks into `CurveValidator` using `soda-core-duckdb`.

**Use this skill in Copilot Chat:**

In [ ]:
'''
/run-soda-programmatic wire soda check into CurveValidator()
'''

✅ **Done when** `CurveValidator()` has a new method to run Soda checks

---
## Step 3 — Run the checks

`CurveValidator` exposes a method that applies the checks programmatically and returns a summary dict. 

**Ask Copilot** to create a notebook so that you can run the checks

In [ ]:
'''
Can you create a notebook so that I can run the Soda checks with CurveValidator()
'''

The created notebook should use one of the curves (like `curve_winter.csv`). All the checks should pass.

Now run against `curve_with_issues.csv` — a synthetic file with known problems (invalid `type`, non-numeric `quote`, missing CET columns). You should see at least 2 non-zero counts:

✅ **Done when** `curve_with_issues.csv` surfaces at least 2 failures.

---
## Step 3 — What SODA catches vs. what the regression catches

| | SODA checks | Regression runner |
|---|---|---|
| Detects | Missing values, invalid types, out-of-range counts | Wrong computed values (e.g. cetStarttime off by 1h) |
| Requires | A YAML config | A reference output CSV |
| Runs | On any dataset | Against a reference output |
| Would have caught the copy-through bug? | No — the value was present, just wrong | Yes — 75 mismatches on first summer run |

> **Discussion**: Both tools are in the pipeline. Which one catches the CET bug? Which one catches the data pipeline sending you garbage input? You need both.

---
## 🎉 All done

> **Reflection questions:**
> 1. You used `/add-soda-checks` on this repo. Could you use the same skill on a different pipeline repo with different column names? What would need to change?
> 2. The SODA check for `missing_count` on `cetStarttime` passes on the buggy input. Why?
> 3. In M3 you wrote a custom prompt. In M5 you used a skill. What was the difference in how you used them?